# Day 45 (2/2) — K-NN from Scratch

K-NN is simple enough to write in a few lines. Doing it yourself makes every step concrete:

1. **Distance** from the new point to every training point
2. **Sort** and take the K smallest
3. **Vote**: the most common label wins

We first check the hand-worked example from **theory §4**, then run our version on the Social Network Ads data and
compare it with scikit-learn.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter

## 1. The building blocks on the tiny example (theory §4)

Six training points, two classes (R = red, B = blue), and a query point Q = (3, 2).

In [2]:
X_toy = np.array([[1, 2], [2, 1], [2, 3], [5, 4], [6, 5], [4, 2]], dtype=float)
y_toy = np.array(['R', 'R', 'R', 'B', 'B', 'B'])
names = ['A', 'B', 'C', 'D', 'E', 'F']
Q = np.array([3, 2])

**Step 1: Euclidean distance** $d = \sqrt{(x_1-q_1)^2 + (x_2-q_2)^2}$ to every point, in one vectorised line:

> ✍️ **Core code: learn this.** Distances, sorting, voting.

In [3]:
distances = np.sqrt(((X_toy - Q) ** 2).sum(axis=1))

table = pd.DataFrame({'point': names, 'x1': X_toy[:, 0], 'x2': X_toy[:, 1], 'class': y_toy,
                      'distance': distances.round(3)})
table.sort_values('distance')      # Step 2: sort by distance

,point,x1,x2,class,distance
5,F,4.0,2.0,B,1.000
1,B,2.0,1.0,R,1.414
2,C,2.0,3.0,R,1.414
0,A,1.0,2.0,R,2.000
3,D,5.0,4.0,B,2.828
4,E,6.0,5.0,B,4.243


In [4]:
# Step 3: vote among the K nearest
order = np.argsort(distances)
for k in [1, 3, 5]:
    nearest_labels = y_toy[order[:k]]
    winner = Counter(nearest_labels).most_common(1)[0][0]
    print(f"K={k}: nearest = {np.array(names)[order[:k]].tolist()}, labels = {nearest_labels.tolist()} -> predict {winner}")

K=1: nearest = ['F'], labels = ['B'] -> predict B
K=3: nearest = ['F', 'B', 'C'], labels = ['B', 'R', 'R'] -> predict R
K=5: nearest = ['F', 'B', 'C', 'A', 'D'], labels = ['B', 'R', 'R', 'R', 'B'] -> predict R


Exactly as in the theory: **K = 1 says Blue, K = 3 and K = 5 say Red.** The choice of K changes the answer, which is
why tuning K matters.

## 2. A reusable `MyKNN` class

> ✍️ **Core code: learn this.** A complete K-NN classifier with the same `fit` / `predict` API as scikit-learn.

In [5]:
class MyKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        # "Training" = remembering the data
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        return np.array([self._predict_one(x) for x in np.asarray(X, dtype=float)])

    def _predict_one(self, x):
        distances = np.sqrt(((self.X_train - x) ** 2).sum(axis=1))   # 1. distance to every training point
        nearest = np.argsort(distances)[:self.k]                       # 2. indices of the K closest
        votes = Counter(self.y_train[nearest])                         # 3. count the labels
        return votes.most_common(1)[0][0]                              #    majority wins

In [6]:
print(MyKNN(k=1).fit(X_toy, y_toy).predict([Q]))
print(MyKNN(k=3).fit(X_toy, y_toy).predict([Q]))

['B']
['R']


## 3. Run it on the Social Network Ads data and compare with scikit-learn

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

dataset = pd.read_csv('Social_Network_Ads.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [8]:
ours = MyKNN(k=5).fit(X_train_sc, y_train).predict(X_test_sc)
sk = KNeighborsClassifier(n_neighbors=5).fit(X_train_sc, y_train).predict(X_test_sc)

print("Our accuracy    :", accuracy_score(y_test, ours))
print("sklearn accuracy:", accuracy_score(y_test, sk))
print("Identical predictions:", (ours == sk).sum(), "of", len(sk))

Our accuracy    : 0.93
sklearn accuracy: 0.93
Identical predictions: 100 of 100


Same predictions as scikit-learn. ✅

### Why use scikit-learn then?
Our version compares each test point with **every** training point: $O(n \cdot d)$ work per prediction. scikit-learn
can build a **KD-tree** or **Ball-tree** index (`algorithm='auto'`) that skips far-away regions, which is much faster
on large, low-dimensional datasets. It also adds distance weighting, other metrics, and `predict_proba`.

---
## Key takeaways
- K-NN = distance → sort → vote. That is the entire algorithm.
- "Training" stores the data; prediction does all the work.
- K changes the answer: tune it with cross-validation.

➡️ Back to [`theory.md`](theory.md) §7–10 for weighted voting, distance metrics, K-NN regression and the curse of dimensionality.